## 빈 components 레코드 제거

`components`가 빈 리스트인 레코드는 재료 정보가 없으므로 제거한다.

In [1]:
from pathlib import Path
import json

GRAPH_INPUT_PATH = next(Path(".").glob("*/recipes_graph_prepared_v2.jsonl"))
GRAPH_OUTPUT_PATH = GRAPH_INPUT_PATH.with_name("recipes_graph_prepared_v2_nonempty.jsonl")

input_records = []
removed_records = 0
with GRAPH_INPUT_PATH.open(encoding="utf-8") as input_file:
    for line_number, line in enumerate(input_file, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"JSON 파싱 실패: {line_number}번째 줄") from error

        if record.get("components") == []:
            removed_records += 1
            continue
        input_records.append(record)

GRAPH_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with GRAPH_OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
    for record in input_records:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"원본 레코드 수: {len(input_records) + removed_records:,}")
print(f"삭제된 레코드 수: {removed_records:,}")
print(f"남은 레코드 수: {len(input_records):,}")
print(f"저장 경로: {GRAPH_OUTPUT_PATH}")

원본 레코드 수: 9,985
삭제된 레코드 수: 430
남은 레코드 수: 9,555
저장 경로: 전처리\recipes_graph_prepared_v2_nonempty.jsonl


In [2]:
# 결과에 components가 빈 리스트인 레코드가 남아 있지 않은지 검증한다.
with GRAPH_OUTPUT_PATH.open(encoding="utf-8") as output_file:
    saved_records = [json.loads(line) for line in output_file if line.strip()]

assert len(saved_records) == len(input_records)
assert all(record.get("components") != [] for record in saved_records)
print("검증 완료: 빈 components 레코드가 제거되었습니다.")

검증 완료: 빈 components 레코드가 제거되었습니다.


## JSONL 데이터 컬럼 이름 확인

두 JSONL 파일의 최상위 컬럼 이름과 그래프 데이터의 중첩 컬럼 이름을 확인한다.

In [ ]:
import json
from pathlib import Path

TOP_VIEWED_PATH = Path("홍기표/input/10000recipe_top_viewed.jsonl")
GRAPH_NONEMPTY_PATH = Path("전처리/recipes_graph_prepared_v2_nonempty.jsonl")

def first_record(path):
    with path.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                return json.loads(line)
    raise ValueError(f"데이터가 비어 있습니다: {path}")

top_viewed_record = first_record(TOP_VIEWED_PATH)
graph_record = first_record(GRAPH_NONEMPTY_PATH)

print(f"[10000recipe_top_viewed.jsonl] 최상위 컬럼 ({len(top_viewed_record)}개)")
print(list(top_viewed_record.keys()))

print(f"\n[recipes_graph_prepared_v2_nonempty.jsonl] 최상위 컬럼 ({len(graph_record)}개)")
print(list(graph_record.keys()))
print("\n중첩 컬럼")
print("recipe:", list(graph_record["recipe"].keys()))
if graph_record.get("components"):
    component = graph_record["components"][0]
    print("components.component:", list(component["component"].keys()))
    print("components.ingredient:", list(component["ingredient"].keys()))
    print("components.alternatives:", list(component["alternat ives"][0].keys()) if component.get("alternatives") else "빈 리스트")

[10000recipe_top_viewed.jsonl] 최상위 컬럼 (12개)
['url', 'rank', 'title', 'author', 'views', 'rating_count', 'description', 'ingredients', 'steps', 'categories', 'source', 'collected_at']

[recipes_graph_prepared_v2_nonempty.jsonl] 최상위 컬럼 (4개)
['schema_version', 'recipe', 'dish', 'components']

중첩 컬럼
recipe: ['recipe_uid', 'title', 'source', 'source_url', 'description', 'servings', 'cooking_time', 'difficulty', 'views']
components.component: ['component_uid', 'raw_name', 'role', 'index', 'alternative_mode', 'evidence', 'quality_flags', 'group', 'quantity', 'unit']
components.ingredient: ['name', 'name_normalized']
components.alternatives: 빈 리스트
